<a href="https://colab.research.google.com/github/Hoor-Fayaz/RehabIQ/blob/datasets/Datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:

import requests

API = "https://dataverse.harvard.edu/api"
DOI = "doi:10.7910/DVN/XSI0QN"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"
}

response = requests.get(
    f"{API}/datasets/:persistentId/",
    params={"persistentId": DOI},
    headers=headers,
    timeout=60
)
response.raise_for_status()

files = response.json()["data"]["latestVersion"]["files"]

# Find E01 video files
videos = []

for item in files:
    f = item["dataFile"]
    name = f["filename"]

    if name.upper().startswith("E01") and name.lower().endswith(".mp4"):
        videos.append({
            "name": name,
            "id": f["id"],
            "size_mb": f.get("filesize", 0) / (1024**2)
        })

print("Shoulder Abduction videos found:", len(videos))

for v in videos[:20]:
    print(f'{v["name"]} | ID={v["id"]} | {v["size_mb"]:.1f} MB')


Shoulder Abduction videos found: 558
E01_P01_AF_VFL_GM.mp4 | ID=11681646 | 71.9 MB
E01_P01_AF_VHJ_GM.mp4 | ID=11681345 | 68.1 MB
E01_P01_AF_VLJ_GM.mp4 | ID=11681677 | 73.0 MB
E01_P01_AF_VLL_GM.mp4 | ID=11681702 | 70.8 MB
E01_P01_AF_VML_GM.mp4 | ID=11681685 | 76.1 MB
E01_P01_AF_VO_GM.mp4 | ID=11681655 | 68.5 MB
E01_P01_AL_VFL_GM.mp4 | ID=11674927 | 23.7 MB
E01_P01_AL_VHJ_GM.mp4 | ID=11679824 | 52.0 MB
E01_P01_AL_VLJ_GM.mp4 | ID=11679838 | 55.3 MB
E01_P01_AL_VLL_GM.mp4 | ID=11681372 | 60.8 MB
E01_P01_AL_VML_GM.mp4 | ID=11674808 | 25.8 MB
E01_P01_AL_VO_GM.mp4 | ID=11674974 | 29.3 MB
E01_P01_AR_VFL_GM.mp4 | ID=11674838 | 37.7 MB
E01_P01_AR_VHJ_GM.mp4 | ID=11681690 | 70.8 MB
E01_P01_AR_VLJ_GM.mp4 | ID=11681699 | 74.3 MB
E01_P01_AR_VLL_GM.mp4 | ID=11681659 | 72.6 MB
E01_P01_AR_VML_GM.mp4 | ID=11674911 | 36.3 MB
E01_P01_AR_VO_GM.mp4 | ID=11681378 | 66.8 MB
E01_P02_AF_VFL_GM.mp4 | ID=11674943 | 12.8 MB
E01_P02_AF_VHJ_GM.mp4 | ID=11674782 | 34.7 MB


In [5]:

from collections import Counter

# Count the filename codes appearing after participant ID.
# Example: E01_P01_AF_VFL_GM.mp4
camera_codes = Counter()
participant_ids = set()

for v in videos:
    parts = v["name"].replace(".mp4", "").split("_")
    if len(parts) >= 5:
        participant_ids.add(parts[1])
        camera_codes[parts[2]] += 1

print("Participants:", len(participant_ids))
print("\nFilename camera/condition codes:")
for code, count in sorted(camera_codes.items()):
    print(f"{code}: {count} videos")


Participants: 51

Filename camera/condition codes:
AF: 262 videos
AL: 150 videos
AR: 146 videos


In [6]:

from google.colab import drive
drive.mount("/content/drive")

import os
import requests
import pandas as pd
from tqdm.auto import tqdm

# Destination folder
OUT_DIR = (
    "/content/drive/MyDrive/RehabIQ_Dataset/"
    "organized_by_exercise/Shoulder_Abduction"
)
os.makedirs(OUT_DIR, exist_ok=True)

# Select frontal-view, full-light videos from different participants
candidates = [
    v for v in videos
    if "_AF_VFL_" in v["name"]
]

# One video per participant
selected = []
seen_participants = set()

for v in candidates:
    participant = v["name"].split("_")[1]

    if participant not in seen_participants:
        selected.append(v)
        seen_participants.add(participant)

    if len(selected) == 5:
        break

print("Selected videos:", len(selected))
for v in selected:
    print(v["name"], f'({v["size_mb"]:.1f} MB)')

# Download each video using the official Dataverse API
session = requests.Session()
downloaded = []

for v in tqdm(selected, desc="Downloading videos"):
    path = os.path.join(OUT_DIR, v["name"])

    if not os.path.exists(path):
        url = f"https://dataverse.harvard.edu/api/access/datafile/{v['id']}"

        with session.get(url, stream=True, timeout=(30, 300)) as r:
            r.raise_for_status()
            with open(path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)

    downloaded.append({
        "filename": v["name"],
        "file_id": v["id"],
        "participant": v["name"].split("_")[1],
        "exercise": "Shoulder Abduction",
        "camera_view": "Front",
        "lighting": "Full light",
        "source": "MobiPhysio",
        "path": path,
        "size_mb": round(os.path.getsize(path) / (1024**2), 2)
    })

# Save the inventory
inventory_path = os.path.join(OUT_DIR, "inventory.csv")
pd.DataFrame(downloaded).to_csv(inventory_path, index=False)

print("\nDownload complete.")
print("Video folder:", OUT_DIR)
print("Inventory:", inventory_path)
print("Videos downloaded:", len(downloaded))


Mounted at /content/drive
Selected videos: 5
E01_P01_AF_VFL_GM.mp4 (71.9 MB)
E01_P02_AF_VFL_GM.mp4 (12.8 MB)
E01_P03_AF_VFL_GM.mp4 (12.0 MB)
E01_P04_AF_VFL_GM.mp4 (17.4 MB)
E01_P05_AF_VFL_GM.mp4 (12.7 MB)



Download complete.
Video folder: /content/drive/MyDrive/RehabIQ_Dataset/organized_by_exercise/Shoulder_Abduction
Inventory: /content/drive/MyDrive/RehabIQ_Dataset/organized_by_exercise/Shoulder_Abduction/inventory.csv
Videos downloaded: 5


In [7]:

import os
import cv2
import pandas as pd

VIDEO_DIR = (
    "/content/drive/MyDrive/RehabIQ_Dataset/"
    "organized_by_exercise/Shoulder_Abduction"
)

results = []

for filename in sorted(os.listdir(VIDEO_DIR)):
    if not filename.lower().endswith(".mp4"):
        continue

    path = os.path.join(VIDEO_DIR, filename)
    cap = cv2.VideoCapture(path)

    opened = cap.isOpened()
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    success, frame = cap.read()
    cap.release()

    duration = frame_count / fps if fps > 0 else 0

    results.append({
        "filename": filename,
        "size_MB": round(os.path.getsize(path) / (1024**2), 2),
        "video_opens": opened,
        "first_frame_readable": success,
        "resolution": f"{width}x{height}",
        "fps": round(fps, 2),
        "frame_count": frame_count,
        "duration_seconds": round(duration, 2)
    })

report = pd.DataFrame(results)
display(report)

print("\nVideos checked:", len(report))
print(
    "All videos readable:",
    len(report) == 5
    and report["video_opens"].all()
    and report["first_frame_readable"].all()
    and (report["frame_count"] > 0).all()
)

report.to_csv(
    os.path.join(VIDEO_DIR, "video_quality_check.csv"),
    index=False
)


,filename,size_MB,video_opens,first_frame_readable,resolution,fps,frame_count,duration_seconds
0,E01_P01_AF_VFL_GM.mp4,71.85,True,True,1080x1920,29.87,909,30.44
1,E01_P02_AF_VFL_GM.mp4,12.81,True,True,1920x1080,24.85,123,4.95
2,E01_P03_AF_VFL_GM.mp4,12.02,True,True,1080x1920,24.42,112,4.59
3,E01_P04_AF_VFL_GM.mp4,17.39,True,True,1080x1920,30.01,758,25.26
4,E01_P05_AF_VFL_GM.mp4,12.70,True,True,1080x1920,30.08,777,25.83



Videos checked: 5
All videos readable: True


In [8]:

from google.colab import drive
drive.mount("/content/drive")

import os
import pandas as pd

ROOT = "/content/drive/MyDrive/RehabIQ_Dataset"

exercises = [
    (1, "Shoulder_Abduction", "MobiPhysio", "E01"),
    (2, "Shoulder_External_Rotation", "MobiPhysio", "Verify code"),
    (3, "Shoulder_Circumduction", "MobiPhysio", "E05"),
    (4, "Arm_Raising", "KIMORE", "Exercise 1"),
    (5, "Hip_Flexion", "MobiPhysio", "E07"),
    (6, "Leg_Abduction", "REHAB24-6", "Exercise 4"),
    (7, "Squat", "REHAB24-6", "Exercise 6"),
    (8, "Seated_Knee_Flexion_Extension", "3D-HPE", "Verify"),
    (9, "Forward_Lunge", "ExeCheck", "Verify"),
    (10, "Lumbar_Flexion", "MobiPhysio", "E08"),
    (11, "Trunk_Lateral_Bending", "KIMORE", "Exercise 2"),
    (12, "Trunk_Rotation", "KIMORE", "Exercise 3"),
]

# Preserve raw source files separately from exercise-organized copies
os.makedirs(os.path.join(ROOT, "raw"), exist_ok=True)
os.makedirs(os.path.join(ROOT, "metadata"), exist_ok=True)
os.makedirs(os.path.join(ROOT, "processed"), exist_ok=True)

for number, name, source, code in exercises:
    os.makedirs(os.path.join(ROOT, "raw", source), exist_ok=True)
    os.makedirs(
        os.path.join(ROOT, "organized_by_exercise", name),
        exist_ok=True
    )

tracker = pd.DataFrame([
    {
        "exercise_number": number,
        "exercise": name,
        "planned_source": source,
        "source_code_or_id": code,
        "videos_collected": 0,
        "movement_verified": "No",
        "labels_verified": "No",
        "status": "Not started"
    }
    for number, name, source, code in exercises
])

# Record the five videos already downloaded
tracker.loc[tracker["exercise_number"] == 1,
            ["videos_collected", "status"]] = [5, "Pilot collected"]

tracker_path = os.path.join(ROOT, "metadata", "collection_tracker.csv")
tracker.to_csv(tracker_path, index=False)

print("Dataset structure created.")
print("Exercise folders:", len(exercises))
print("Tracker:", tracker_path)
display(tracker)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset structure created.
Exercise folders: 12
Tracker: /content/drive/MyDrive/RehabIQ_Dataset/metadata/collection_tracker.csv


,exercise_number,exercise,planned_source,source_code_or_id,videos_collected,movement_verified,labels_verified,status
0,1,Shoulder_Abduction,MobiPhysio,E01,5,No,No,Pilot collected
1,2,Shoulder_External_Rotation,MobiPhysio,Verify code,0,No,No,Not started
2,3,Shoulder_Circumduction,MobiPhysio,E05,0,No,No,Not started
3,4,Arm_Raising,KIMORE,Exercise 1,0,No,No,Not started
4,5,Hip_Flexion,MobiPhysio,E07,0,No,No,Not started
5,6,Leg_Abduction,REHAB24-6,Exercise 4,0,No,No,Not started
6,7,Squat,REHAB24-6,Exercise 6,0,No,No,Not started
7,8,Seated_Knee_Flexion_Extension,3D-HPE,Verify,0,No,No,Not started
8,9,Forward_Lunge,ExeCheck,Verify,0,No,No,Not started
9,10,Lumbar_Flexion,MobiPhysio,E08,0,No,No,Not started


In [12]:
import os
import time
import requests
import pandas as pd
from tqdm.auto import tqdm

API = "https://dataverse.harvard.edu/api"
DOI = "doi:10.7910/DVN/XSI0QN"

ROOT = "/content/drive/MyDrive/RehabIQ_Dataset"
OUT_DIR = os.path.join(
    ROOT, "organized_by_exercise", "Shoulder_Abduction"
)
META_DIR = os.path.join(ROOT, "metadata")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(META_DIR, exist_ok=True)

# Standard user agent header to prevent 403 Forbidden errors
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"
}

# 1. Get the current Dataverse file inventory
response = requests.get(
    f"{API}/datasets/:persistentId/",
    params={"persistentId": DOI},
    headers=headers,
    timeout=60
)
response.raise_for_status()
all_files = response.json()["data"]["latestVersion"]["files"]

# 2. Select E01 front-view MP4 videos
candidates = []

for item in all_files:
    f = item["dataFile"]
    name = f["filename"]
    parts = name.replace(".mp4", "").split("_")

    if (
        len(parts) == 5
        and parts[0] == "E01"
        and parts[2] == "AF"
        and name.lower().endswith(".mp4")
    ):
        candidates.append({
            "filename": name,
            "file_id": f["id"],
            "size_bytes": f.get("filesize", 0),
            "participant": parts[1],
            "angle_code": parts[2],
            "variation_code": parts[3],
            "gender_code": parts[4]
        })

candidates.sort(key=lambda x: x["filename"])

manifest_path = os.path.join(
    META_DIR, "shoulder_abduction_front_manifest.csv"
)

manifest = pd.DataFrame(candidates)
manifest["source"] = "MobiPhysio"
manifest["exercise"] = "Shoulder Abduction"
manifest["download_path"] = manifest["filename"].apply(
    lambda name: os.path.join(OUT_DIR, name)
)
manifest.to_csv(manifest_path, index=False)

total_gb = sum(v["size_bytes"] for v in candidates) / (1024**3)

print("Front-view videos found:", len(candidates))
print("Unique participants:", manifest["participant"].nunique())
print(f"Total listed video size: {total_gb:.2f} GB")
print("Manifest saved to:", manifest_path)

# Check that the selection is the expected size before proceeding
if not candidates:
    raise RuntimeError("No videos found. Check the filename convention.")

confirm = input(
    "If you have enough Google Drive space, type DOWNLOAD to proceed: "
).strip()

if confirm != "DOWNLOAD":
    print("Download cancelled. The manifest is still saved.")
else:
    session = requests.Session()
    session.headers.update(headers)
    completed = []
    failed = []

    for v in tqdm(candidates, desc="Downloading E01 front-view videos"):
        final_path = os.path.join(OUT_DIR, v["filename"])
        partial_path = final_path + ".part"

        # Preserve files already downloaded
        if os.path.exists(final_path) and os.path.getsize(final_path) > 0:
            completed.append(v["filename"])
            continue

        url = f"{API}/access/datafile/{v["file_id"]}"
        success = False

        for attempt in range(3):
            try:
                with session.get(
                    url, stream=True, timeout=(30, 300)
                ) as r:
                    r.raise_for_status()
                    with open(partial_path, "wb") as f:
                        for chunk in r.iter_content(
                            chunk_size=1024 * 1024
                        ):
                            if chunk:
                                f.write(chunk)

                # Basic size check against Dataverse metadata
                actual_size = os.path.getsize(partial_path)
                expected_size = v["size_bytes"]

                if expected_size and actual_size != expected_size:
                    raise ValueError(
                        f"Size mismatch: expected {expected_size}, "
                        f"got {actual_size}"
                    )

                os.replace(partial_path, final_path)
                completed.append(v["filename"])
                success = True
                break

            except Exception as e:
                print(
                    f"\nAttempt {attempt + 1}/3 failed for "
                    f"{v["filename"]}: {e}"
                )
                time.sleep(2 * (attempt + 1))

        if not success:
            if os.path.exists(partial_path):
                os.remove(partial_path)
            failed.append({
                "filename": v["filename"],
                "file_id": v["file_id"]
            })

        # Save progress in case Colab disconnects
        pd.DataFrame({
            "downloaded_filename": completed
        }).to_csv(
            os.path.join(META_DIR, "shoulder_abduction_downloaded.csv"),
            index=False
        )

        pd.DataFrame(failed).to_csv(
            os.path.join(META_DIR, "shoulder_abduction_failed.csv"),
            index=False
        )

    print("\nDownload run finished.")
    print("Successfully present:", len(completed))
    print("Failed downloads:", len(failed))
    print("Video folder:", OUT_DIR)
    print("Manifest:", manifest_path)

Front-view videos found: 262
Unique participants: 51
Total listed video size: 5.38 GB
Manifest saved to: /content/drive/MyDrive/RehabIQ_Dataset/metadata/shoulder_abduction_front_manifest.csv
If you have enough Google Drive space, type DOWNLOAD to proceed: DOWNLOAD



Download run finished.
Successfully present: 262
Failed downloads: 0
Video folder: /content/drive/MyDrive/RehabIQ_Dataset/organized_by_exercise/Shoulder_Abduction
Manifest: /content/drive/MyDrive/RehabIQ_Dataset/metadata/shoulder_abduction_front_manifest.csv


In [14]:

import os
import requests

API = "https://dataverse.harvard.edu/api"
DOI = "doi:10.7910/DVN/XSI0QN"

ROOT = "/content/drive/MyDrive/RehabIQ_Dataset"
META_DIR = os.path.join(ROOT, "metadata")
os.makedirs(META_DIR, exist_ok=True)

# Standard user agent header to prevent 403 Forbidden errors
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"
}

response = requests.get(
    f"{API}/datasets/:persistentId/",
    params={"persistentId": DOI},
    headers=headers,
    timeout=60
)
response.raise_for_status()
files = response.json()["data"]["latestVersion"]["files"]

# First, list metadata files so we can identify the official score file.
print("=== CSV / README / DOCUMENTATION FILES ===")
metadata_files = []

for item in files:
    f = item["dataFile"]
    name = f["filename"]
    lower = name.lower()

    if lower.endswith((".csv", ".txt", ".md", ".readme", ".pdf")):
        metadata_files.append(f)
        print(
            f'{name} | ID={f["id"]} | '
            f'{f.get("filesize", 0) / (1024**2):.2f} MB'
        )

# Download only small metadata files; skip PDFs and large files for now.
session = requests.Session()
session.headers.update(headers)
downloaded_metadata = []

for f in metadata_files:
    name = f["filename"]
    lower = name.lower()
    size = f.get("filesize", 0)

    is_relevant = (
        lower.endswith((".csv", ".txt", ".md", ".readme"))
    )

    if not is_relevant or size > 25 * 1024 * 1024:
        continue

    path = os.path.join(META_DIR, name)

    if not os.path.exists(path):
        url = f'{API}/access/datafile/{f["id"]}'
        r = session.get(url, timeout=120)
        r.raise_for_status()

        with open(path, "wb") as out:
            out.write(r.content)

    downloaded_metadata.append(path)
    print("Saved:", path)

print("\nMetadata files downloaded or already present:",
      len(downloaded_metadata))


=== CSV / README / DOCUMENTATION FILES ===
MobiPhysio.md | ID=13453294 | 0.01 MB
Physiotherapist Exercise Marking .csv | ID=13453292 | 0.51 MB
Saved: /content/drive/MyDrive/RehabIQ_Dataset/metadata/MobiPhysio.md
Saved: /content/drive/MyDrive/RehabIQ_Dataset/metadata/Physiotherapist Exercise Marking .csv

Metadata files downloaded or already present: 2


In [15]:

import os
import pandas as pd

ROOT = "/content/drive/MyDrive/RehabIQ_Dataset"
META_DIR = os.path.join(ROOT, "metadata")

readme_path = os.path.join(META_DIR, "MobiPhysio.md")
csv_path = os.path.join(META_DIR, "Physiotherapist Exercise Marking .csv")

# 1. Read dataset documentation
print("=" * 70)
print("MOBIPHYSIO DOCUMENTATION")
print("=" * 70)

with open(readme_path, "r", encoding="utf-8") as f:
    readme = f.read()

print(readme[:12000])

# 2. Inspect assessment CSV
print("\n" + "=" * 70)
print("ASSESSMENT CSV")
print("=" * 70)

df = pd.read_csv(csv_path)

print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 10 rows:")
display(df.head(10))

print("\nData types and missing values:")
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique_values": df.nunique(dropna=True)
})
display(summary)

# 3. Look for fields that may identify participants, exercises or videos
print("\nPotential identifier / assessment columns:")
for col in df.columns:
    if any(word in col.lower() for word in [
        "file", "video", "participant", "subject", "exercise",
        "score", "mark", "label", "correct", "id", "camera"
    ]):
        print("-", col)

# Save an inspection copy for convenient review
df.head(100).to_csv(
    os.path.join(META_DIR, "assessment_preview.csv"),
    index=False
)

print("\nInspection complete. No assessment labels have been changed.")


MOBIPHYSIO DOCUMENTATION
# MobiPhysio Dataset (v1.1)

MobiPhysio is a **2D smartphone video dataset of physiotherapy exercises** designed to support research and development of **AI-driven exercise assessment and monitoring** using low-cost 2D camera devices (e.g., mobile phones). It includes multi-angle recordings and real-world variations (lighting, jitter, occlusion, and resolution), along with **expert physiotherapist evaluation scores**.

## Quick facts (from the Excel summary sheet)
- **Total videos:** 3,686
- **Participants:** 58 (**Male:** 39, **Female:** 19)
- **Exercises:** 9 Active Range of Motion (AROM) physiotherapy exercises (E01–E09)
- **Angles:** Front / Left / Right
- **Annotations:** Exercise Accuracy Assessment Questionnaire (EAAQ) scores by **three certified physiotherapists** (averaged and scaled to 0–100)

## Repository / DOI
Dataset DOI:
```text
doi:10.7910/DVN/XSI0QN
```

## Associated publication
Please refer to the accompanying **Data in Brief** article/manusc

,Video Name,Judges,Primary Factors,Unnamed: 3,Unnamed: 4,Unnamed: 5,Control Factors,Unnamed: 7,Unnamed: 8,Assesment Score,Unnamed: 10,Unnamed: 11,Unnamed: 12
0,NaN,NaN,1.0,2.0,3.0,4.0,1.0,2.0,3.0,Total Mark,On scale of 100,Average Score,Rounding to 2 Decimal
1,E01_P01_AF_VFL_GM,Dr Mamun Mahmud,1.0,1.0,0.0,0.9,1.0,1.0,0.0,4.9,70,79.52380952,79.52
2,NaN,Dr Abid Hasan Khan,1.0,1.0,0.0,0.9,1.0,1.0,1.0,5.9,84.28571429,NaN,NaN
3,NaN,Dr Rayhan Islam,1.0,1.0,0.0,0.9,1.0,1.0,1.0,5.9,84.28571429,NaN,NaN
4,E01_P01_AL_VFL_GM,Dr Mamun Mahmud,1.0,1.0,0.0,0.8,1.0,1.0,0.0,4.8,68.57142857,78.0952381,78.1
5,NaN,Dr Abid Hasan Khan,1.0,1.0,0.0,0.8,1.0,1.0,1.0,5.8,82.85714286,NaN,NaN
6,NaN,Dr Rayhan Islam,1.0,1.0,0.0,0.8,1.0,1.0,1.0,5.8,82.85714286,NaN,NaN
7,E01_P01_AR_VFL_GM,Dr Mamun Mahmud,1.0,1.0,0.0,0.8,1.0,1.0,0.0,4.8,68.57142857,78.0952381,78.1
8,NaN,Dr Abid Hasan Khan,1.0,1.0,0.0,0.8,1.0,1.0,1.0,5.8,82.85714286,NaN,NaN
9,NaN,Dr Rayhan Islam,1.0,1.0,0.0,0.8,1.0,1.0,1.0,5.8,82.85714286,NaN,NaN



Data types and missing values:


,dtype,missing,unique_values
Video Name,object,6055,3010
Judges,object,32,3
Primary Factors,float64,31,2
Unnamed: 3,float64,31,4
Unnamed: 4,float64,31,3
Unnamed: 5,float64,31,12
Control Factors,float64,31,3
Unnamed: 7,float64,31,4
Unnamed: 8,float64,31,6
Assesment Score,object,31,45



Potential identifier / assessment columns:
- Video Name
- Assesment Score

Inspection complete. No assessment labels have been changed.


In [16]:

import os
import pandas as pd

ROOT = "/content/drive/MyDrive/RehabIQ_Dataset"
META_DIR = os.path.join(ROOT, "metadata")
VIDEO_DIR = os.path.join(
    ROOT, "organized_by_exercise", "Shoulder_Abduction"
)

csv_path = os.path.join(
    META_DIR, "Physiotherapist Exercise Marking .csv"
)

# The exported CSV has two header rows; inspect them before parsing.
raw = pd.read_csv(csv_path, header=None)

print("Raw shape:", raw.shape)
display(raw.head(8))

# Use the first row as the top-level header, then make unique names.
df = pd.read_csv(csv_path, header=0)
df.columns = [
    str(c).strip() if not str(c).startswith("Unnamed") else f"column_{i}"
    for i, c in enumerate(df.columns)
]

print("\nColumns:", df.columns.tolist())
display(df.head(8))

# Use only rows with a video name to identify annotation-group starts.
# Do not fill down and average blindly: first inspect group boundaries.
named = df[df["Video Name"].notna()].copy()
named["Video Name"] = named["Video Name"].astype(str).str.strip()

print("\nNamed annotation rows:", len(named))
print("Unique named videos:", named["Video Name"].nunique())
print("\nSample names:")
display(named[["Video Name"]].head(15))

# Check how many of our downloaded videos have an exact annotation name.
video_names = {
    os.path.splitext(f)[0]
    for f in os.listdir(VIDEO_DIR)
    if f.lower().endswith(".mp4")
}

annotation_names = set(named["Video Name"])
matched = video_names & annotation_names

print("\nDownloaded videos:", len(video_names))
print("Exact name matches:", len(matched))
print("Videos without exact annotation match:", len(video_names - matched))

print("\nExamples of unmatched names:")
print(sorted(video_names - matched)[:10])


Raw shape: (9066, 13)


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,Video Name,Judges,Primary Factors,NaN,NaN,NaN,Control Factors,NaN,NaN,Assesment Score,NaN,NaN,NaN
1,NaN,NaN,1,2.0,3.0,4.0,1,2.0,3.0,Total Mark,On scale of 100,Average Score,Rounding to 2 Decimal
2,E01_P01_AF_VFL_GM,Dr Mamun Mahmud,1,1.0,0.0,0.9,1,1.0,0.0,4.9,70,79.52380952,79.52
3,NaN,Dr Abid Hasan Khan,1,1.0,0.0,0.9,1,1.0,1.0,5.9,84.28571429,NaN,NaN
4,NaN,Dr Rayhan Islam,1,1.0,0.0,0.9,1,1.0,1.0,5.9,84.28571429,NaN,NaN
5,E01_P01_AL_VFL_GM,Dr Mamun Mahmud,1,1.0,0.0,0.8,1,1.0,0.0,4.8,68.57142857,78.0952381,78.1
6,NaN,Dr Abid Hasan Khan,1,1.0,0.0,0.8,1,1.0,1.0,5.8,82.85714286,NaN,NaN
7,NaN,Dr Rayhan Islam,1,1.0,0.0,0.8,1,1.0,1.0,5.8,82.85714286,NaN,NaN



Columns: ['Video Name', 'Judges', 'Primary Factors', 'column_3', 'column_4', 'column_5', 'Control Factors', 'column_7', 'column_8', 'Assesment Score', 'column_10', 'column_11', 'column_12']


,Video Name,Judges,Primary Factors,column_3,column_4,column_5,Control Factors,column_7,column_8,Assesment Score,column_10,column_11,column_12
0,NaN,NaN,1.0,2.0,3.0,4.0,1.0,2.0,3.0,Total Mark,On scale of 100,Average Score,Rounding to 2 Decimal
1,E01_P01_AF_VFL_GM,Dr Mamun Mahmud,1.0,1.0,0.0,0.9,1.0,1.0,0.0,4.9,70,79.52380952,79.52
2,NaN,Dr Abid Hasan Khan,1.0,1.0,0.0,0.9,1.0,1.0,1.0,5.9,84.28571429,NaN,NaN
3,NaN,Dr Rayhan Islam,1.0,1.0,0.0,0.9,1.0,1.0,1.0,5.9,84.28571429,NaN,NaN
4,E01_P01_AL_VFL_GM,Dr Mamun Mahmud,1.0,1.0,0.0,0.8,1.0,1.0,0.0,4.8,68.57142857,78.0952381,78.1
5,NaN,Dr Abid Hasan Khan,1.0,1.0,0.0,0.8,1.0,1.0,1.0,5.8,82.85714286,NaN,NaN
6,NaN,Dr Rayhan Islam,1.0,1.0,0.0,0.8,1.0,1.0,1.0,5.8,82.85714286,NaN,NaN
7,E01_P01_AR_VFL_GM,Dr Mamun Mahmud,1.0,1.0,0.0,0.8,1.0,1.0,0.0,4.8,68.57142857,78.0952381,78.1



Named annotation rows: 3010
Unique named videos: 3010

Sample names:


,Video Name
1,E01_P01_AF_VFL_GM
4,E01_P01_AL_VFL_GM
7,E01_P01_AR_VFL_GM
10,E01_P01_AF_VHJ_GM
13,E01_P01_AL_VHJ_GM
16,E01_P01_AR_VHJ_GM
19,E01_P01_AF_VLJ_GM
22,E01_P01_AL_VLJ_GM
28,E01_P01_AF_VLL_GM
31,E01_P01_AL_VLL_GM



Downloaded videos: 262
Exact name matches: 128
Videos without exact annotation match: 134

Examples of unmatched names:
['E01_P26_AF_VFL_GF', 'E01_P26_AF_VHJ_GF', 'E01_P26_AF_VLJ_GF', 'E01_P26_AF_VLL_GF', 'E01_P26_AF_VLR_GF', 'E01_P27_AF_VFL_GM', 'E01_P27_AF_VHJ_GM', 'E01_P27_AF_VLJ_GM', 'E01_P27_AF_VLL_GM', 'E01_P27_AF_VML_GM']


In [17]:

import os
import pandas as pd

ROOT = "/content/drive/MyDrive/RehabIQ_Dataset"
META_DIR = os.path.join(ROOT, "metadata")
VIDEO_DIR = os.path.join(
    ROOT, "organized_by_exercise", "Shoulder_Abduction"
)

csv_path = os.path.join(
    META_DIR, "Physiotherapist Exercise Marking .csv"
)

# Load the two-row header structure
raw = pd.read_csv(csv_path, header=None)

# Data begins after the two header rows.
data = raw.iloc[2:].copy()
data.columns = [
    "Video Name", "Judge",
    "Primary_1", "Primary_2", "Primary_3", "Primary_4",
    "Control_1", "Control_2", "Control_3",
    "Total_Mark", "Score_100", "Average_Score", "Rounded_Score"
]

# Keep the original score rows; fill video name only to identify its group.
data["Video Name"] = data["Video Name"].replace(r"^\s*$", pd.NA, regex=True)
data["Video Name"] = data["Video Name"].ffill()

# Select one record per video, using the provided average/rounded score
# stored on the first judge row for that video.
labels = data.drop_duplicates("Video Name", keep="first").copy()

labels["Average_Score"] = pd.to_numeric(
    labels["Average_Score"], errors="coerce"
)
labels["Rounded_Score"] = pd.to_numeric(
    labels["Rounded_Score"], errors="coerce"
)

labels = labels[[
    "Video Name", "Average_Score", "Rounded_Score"
]].rename(columns={
    "Average_Score": "average_score_0_100",
    "Rounded_Score": "rounded_score_0_100"
})

# Build a table of actual downloaded video files.
video_files = [
    f for f in os.listdir(VIDEO_DIR)
    if f.lower().endswith(".mp4")
]

downloaded = pd.DataFrame({
    "filename": video_files
})
downloaded["Video Name"] = downloaded["filename"].str.replace(
    ".mp4", "", regex=False
)

# Left join so videos without labels remain in the table.
dataset = downloaded.merge(labels, on="Video Name", how="left")

dataset["label_available"] = dataset["average_score_0_100"].notna()

output_path = os.path.join(
    META_DIR, "shoulder_abduction_dataset.csv"
)
dataset.to_csv(output_path, index=False)

print("Downloaded videos:", len(dataset))
print("Matched scores:", int(dataset["label_available"].sum()))
print("Missing scores:", int((~dataset["label_available"]).sum()))
print("Saved:", output_path)

display(dataset.head(10))


Downloaded videos: 262
Matched scores: 128
Missing scores: 134
Saved: /content/drive/MyDrive/RehabIQ_Dataset/metadata/shoulder_abduction_dataset.csv


,filename,Video Name,average_score_0_100,rounded_score_0_100,label_available
0,E01_P01_AF_VFL_GM.mp4,E01_P01_AF_VFL_GM,79.523810,79.52,True
1,E01_P02_AF_VFL_GM.mp4,E01_P02_AF_VFL_GM,92.857143,92.86,True
2,E01_P03_AF_VFL_GM.mp4,E01_P03_AF_VFL_GM,60.476190,60.48,True
3,E01_P04_AF_VFL_GM.mp4,E01_P04_AF_VFL_GM,82.857143,82.86,True
4,E01_P05_AF_VFL_GM.mp4,E01_P05_AF_VFL_GM,79.523810,79.52,True
5,E01_P01_AF_VHJ_GM.mp4,E01_P01_AF_VHJ_GM,77.142857,77.14,True
6,E01_P01_AF_VLJ_GM.mp4,E01_P01_AF_VLJ_GM,77.142857,77.14,True
7,E01_P01_AF_VLL_GM.mp4,E01_P01_AF_VLL_GM,77.142857,77.14,True
8,E01_P01_AF_VML_GM.mp4,E01_P01_AF_VML_GM,100.000000,100.00,True
9,E01_P01_AF_VO_GM.mp4,E01_P01_AF_VO_GM,74.761905,74.76,True


In [19]:
import os
import pandas as pd

ROOT = "/content/drive/MyDrive/RehabIQ_Dataset"
META_DIR = os.path.join(ROOT, "metadata")

dataset_path = os.path.join(
    META_DIR, "shoulder_abduction_dataset.csv"
)
annotations_path = os.path.join(
    META_DIR, "Physiotherapist Exercise Marking .csv"
)

dataset = pd.read_csv(dataset_path)
raw = pd.read_csv(annotations_path, header=None)

# Extract annotation video names from the official CSV
annotation_names = (
    raw.iloc[2:, 0]
    .dropna()
    .astype(str)
    .str.strip()
)

annotation_names = annotation_names[
    annotation_names.ne("")
]

# Identify unmatched downloaded videos
missing = dataset[~dataset["label_available"]].copy()

missing["participant"] = missing["filename"].str.extract(
    r"_(P\d+)_"
)
missing["variation"] = missing["filename"].str.extract(
    r"_(V[^_]+)_"
)

missing["name_appears_in_csv"] = (
    missing["Video Name"].isin(set(annotation_names))
)

print("Total downloaded:", len(dataset))
print("Matched scores:", int(dataset["label_available"].sum()))
print("Missing scores:", len(missing))

print("\nMissing videos by participant:")
print(missing["participant"].value_counts().sort_index().to_string())

print("\nMissing videos by variation:")
print(missing["variation"].value_counts().to_string())

print("\nMissing videos whose names appear in the raw CSV:")
print(missing["name_appears_in_csv"].value_counts().to_string())

print("\nFirst 30 unmatched filenames:")
print(missing["filename"].head(30).to_string(index=False))

print("\nSaved diagnostic report:")
diagnostic_path = os.path.join(
    META_DIR, "shoulder_abduction_missing_score_diagnostic.csv"
)
missing.to_csv(diagnostic_path, index=False)
print(diagnostic_path)

Total downloaded: 262
Matched scores: 128
Missing scores: 134

Missing videos by participant:
participant
P26    5
P27    5
P28    7
P29    7
P30    6
P31    7
P32    7
P33    4
P34    6
P35    7
P36    7
P37    1
P39    1
P40    6
P41    1
P42    1
P48    2
P52    7
P53    7
P55    6
P56    4
P57    4
P58    6
P59    5
P60    7
P61    2
P62    6

Missing videos by variation:
variation
VLJ    22
VFL    21
VLL    21
VHJ    20
VML    19
VO     16
VLR    15

Missing videos whose names appear in the raw CSV:
name_appears_in_csv
False    134

First 30 unmatched filenames:
E01_P26_AF_VFL_GF.mp4
E01_P26_AF_VHJ_GF.mp4
E01_P26_AF_VLJ_GF.mp4
E01_P26_AF_VLL_GF.mp4
E01_P26_AF_VLR_GF.mp4
E01_P27_AF_VFL_GM.mp4
E01_P27_AF_VHJ_GM.mp4
E01_P27_AF_VLJ_GM.mp4
E01_P27_AF_VLL_GM.mp4
E01_P27_AF_VML_GM.mp4
E01_P28_AF_VFL_GM.mp4
E01_P28_AF_VHJ_GM.mp4
E01_P28_AF_VLJ_GM.mp4
E01_P28_AF_VLL_GM.mp4
E01_P28_AF_VLR_GM.mp4
E01_P28_AF_VML_GM.mp4
 E01_P28_AF_VO_GM.mp4
E01_P29_AF_VFL_GM.mp4
E01_P29_AF_VHJ_GM.mp4
E01_P29_

In [21]:
import requests
import pandas as pd

API = "https://dataverse.harvard.edu/api"
DOI = "doi:10.7910/DVN/XSI0QN"

# Standard user agent header to prevent 403 Forbidden errors
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"
}

response = requests.get(
    f"{API}/datasets/:persistentId/",
    params={"persistentId": DOI},
    headers=headers,
    timeout=60
)
response.raise_for_status()

files = response.json()["data"]["latestVersion"]["files"]

inventory = []
for item in files:
    datafile = item.get("dataFile", {})
    inventory.append({
        "filename": item.get("label", ""),
        "file_id": datafile.get("id"),
        "size_MB": round(datafile.get("filesize", 0) / (1024 * 1024), 2),
        "content_type": datafile.get("contentType", "")
    })

inventory_df = pd.DataFrame(inventory)

print("Total files in repository:", len(inventory_df))

print("\nPotential annotation, score, or documentation files:")
keywords = r"score|mark|annotat|assessment|combine|readme|metadata|label"
print(
    inventory_df[
        inventory_df["filename"].str.contains(
            keywords, case=False, na=False
        )
    ].to_string(index=False)
)

inventory_df.to_csv(
    "/content/drive/MyDrive/RehabIQ_Dataset/metadata/"
    "mobiphysio_repository_inventory.csv",
    index=False
)

Total files in repository: 3688

Potential annotation, score, or documentation files:
                             filename  file_id  size_MB                content_type
Physiotherapist Exercise Marking .csv 13453292     0.51 text/comma-separated-values


In [22]:
# Inspect all non-video files in the MobiPhysio repository
non_video = inventory_df[
    ~inventory_df["filename"].str.lower().str.endswith(
        (".mp4", ".avi", ".mov", ".mkv")
    )
].copy()

print("Non-video files:", len(non_video))
print("\nNon-video filenames:")
print(non_video[["filename", "file_id", "size_MB"]].to_string(index=False))

print("\nPossible spreadsheets or structured metadata:")
extensions = (
    ".csv", ".xlsx", ".xls", ".json", ".xml",
    ".md", ".txt", ".zip", ".pdf"
)

structured = non_video[
    non_video["filename"].str.lower().str.endswith(extensions)
]

print(structured[["filename", "file_id", "size_MB"]].to_string(index=False))

Non-video files: 2

Non-video filenames:
                             filename  file_id  size_MB
                        MobiPhysio.md 13453294     0.01
Physiotherapist Exercise Marking .csv 13453292     0.51

Possible spreadsheets or structured metadata:
                             filename  file_id  size_MB
                        MobiPhysio.md 13453294     0.01
Physiotherapist Exercise Marking .csv 13453292     0.51


In [23]:
import os
import pandas as pd

ROOT = "/content/drive/MyDrive/RehabIQ_Dataset"
META = os.path.join(ROOT, "metadata")
VIDEO_DIR = os.path.join(
    ROOT, "organized_by_exercise", "Shoulder_Abduction"
)

dataset_path = os.path.join(
    META, "shoulder_abduction_dataset.csv"
)
tracker_path = os.path.join(META, "collection_tracker.csv")

# Load the dataset manifest
df = pd.read_csv(dataset_path)

# Verify files physically exist and are non-empty
df["file_exists"] = df["filename"].apply(
    lambda name: os.path.isfile(os.path.join(VIDEO_DIR, name))
)
df["file_size_bytes"] = df["filename"].apply(
    lambda name: (
        os.path.getsize(os.path.join(VIDEO_DIR, name))
        if os.path.isfile(os.path.join(VIDEO_DIR, name))
        else 0
    )
)

# Summary
print("EXERCISE 1: SHOULDER ABDUCTION")
print("-" * 40)
print("Manifest entries:", len(df))
print("Files present:", int(df["file_exists"].sum()))
print("Missing files:", int((~df["file_exists"]).sum()))
print("Empty files:", int((df["file_size_bytes"] == 0).sum()))
print("Duplicate filenames:", int(df["filename"].duplicated().sum()))
print("Videos with official score matches:",
      int(df["label_available"].sum()))
print("Videos without matched scores:",
      int((~df["label_available"]).sum()))
print("Total video storage (GB):",
      round(df["file_size_bytes"].sum() / (1024**3), 2))

# Save updated quality-check report
report_path = os.path.join(
    META, "shoulder_abduction_quality_check.csv"
)
df.to_csv(report_path, index=False)
print("\nQuality report saved to:", report_path)

# Update the tracker if the exercise row exists
if os.path.exists(tracker_path):
    tracker = pd.read_csv(tracker_path)

    exercise_col = next(
        (c for c in tracker.columns
         if c.strip().lower() == "exercise"),
        None
    )

    if exercise_col:
        mask = tracker[exercise_col].astype(str).str.contains(
            "shoulder abduction", case=False, na=False
        )

        if mask.any():
            for col in tracker.columns:
                key = col.strip().lower().replace(" ", "_")
                if key in ["videos_collected", "video_count",
                           "videos_downloaded"]:
                    tracker.loc[mask, col] = len(df)

                elif key in ["status", "collection_status"]:
                    tracker.loc[mask, col] = (
                        "Collected - score coverage incomplete"
                    )

            tracker.to_csv(tracker_path, index=False)
            print("Collection tracker updated:", tracker_path)
        else:
            print("Tracker row not found; tracker left unchanged.")
    else:
        print("Tracker schema differs; tracker left unchanged.")
else:
    print("Tracker not found; quality report still saved.")

EXERCISE 1: SHOULDER ABDUCTION
----------------------------------------
Manifest entries: 262
Files present: 262
Missing files: 0
Empty files: 0
Duplicate filenames: 0
Videos with official score matches: 128
Videos without matched scores: 134
Total video storage (GB): 5.38

Quality report saved to: /content/drive/MyDrive/RehabIQ_Dataset/metadata/shoulder_abduction_quality_check.csv
Tracker row not found; tracker left unchanged.
